In [0]:
from pyspark.sql.functions import *

rows = 10002

df = (
    spark.range(rows)

    # IDs
    .withColumn("TransactionID", concat(lit("TXN"), lpad(col("id"),6,"0")))
    .withColumn("EncounterID", concat(lit("ENC"), lpad((rand()*999999).cast("int"),6,"0")))
    .withColumn("PatientID", concat(lit("PAT"), lpad((rand()*99999).cast("int"),5,"0")))
    .withColumn("ProviderID", concat(lit("PROV"), lpad((rand()*500).cast("int"),4,"0")))
    .withColumn("DeptID", concat(lit("DEPT"), lpad((rand()*20).cast("int"),3,"0")))

    # Dates
    .withColumn("VisitDate", date_format(date_sub(current_date(), (rand()*365).cast("int")), "M/d/yyyy"))
    .withColumn("ServiceDate", date_format(date_sub(current_date(), (rand()*365).cast("int")), "M/d/yyyy"))
    .withColumn("PaidDate", date_format(date_sub(current_date(), (rand()*365).cast("int")), "M/d/yyyy"))

    # Visit type
    .withColumn("VisitType",
        expr("CASE WHEN rand()<0.25 THEN 'Routine' \
                   WHEN rand()<0.5 THEN 'Emergency' \
                   WHEN rand()<0.75 THEN 'Consultation' \
                   ELSE 'Follow-up' END")
    )

    # Amounts
    .withColumn("Amount", rand()*1000)
    .withColumn("AmountType",
        expr("CASE WHEN rand()<0.33 THEN 'Insurance' \
                   WHEN rand()<0.66 THEN 'Medicare' \
                   ELSE 'Co-pay' END")
    )
    .withColumn("PaidAmount", rand()*800)

    # Claim / Payor
    .withColumn("ClaimID", concat(lit("CLAIM"), lpad((rand()*999999).cast("int"),6,"0")))
    .withColumn("PayorID", concat(lit("PAYOR"), lpad((rand()*9999).cast("int"),4,"0")))

    # Codes
    .withColumn("ProcedureCode", lpad((rand()*99999).cast("int"),5,"0"))
    .withColumn("ICDCode", concat(lit("I"), (rand()*90).cast("int"), lit("."), (rand()*9).cast("int")))

    # Line of business
    .withColumn("LineOfBusiness",
        expr("CASE WHEN rand()<0.33 THEN 'Commercial' \
                   WHEN rand()<0.66 THEN 'Medicaid' \
                   ELSE 'Self-Pay' END")
    )

    # IDs
    .withColumn("MedicaidID", concat(lit("MEDI"), lpad((rand()*99999).cast("int"),5,"0")))
    .withColumn("MedicareID", concat(lit("MCARE"), lpad((rand()*99999).cast("int"),5,"0")))

    # Insert / Modified dates
    .withColumn("InsertDate", date_format(date_sub(current_date(), (rand()*1500).cast("int")), "M/d/yyyy"))
    .withColumn("ModifiedDate", date_format(date_sub(current_date(), (rand()*1500).cast("int")), "M/d/yyyy"))

    .drop("id")
)

#df.display()

In [0]:
df.count()

In [0]:
df.limit(2).display()

In [0]:
# from pyspark.sql.functions import *

# # Count rows in the DataFrame
# row_count = df.count()
# print(f"Total rows in df: {row_count:,}")

In [0]:
# path = "/Volumes/accenture/manishgautam/manishvolume/structuredStreaming/src"

# # write file
# df.coalesce(1).write.mode("overwrite").option("header",True).csv(path)

In [0]:
from datetime import datetime

base_path = "/Volumes/accenture/manishgautam/manishvolume/structuredStreaming/src"
temp_path = base_path + "/archive"

file_name = "dataset_2"

# write to temporary folder
df.coalesce(1).write.mode("overwrite").option("header", True).csv(temp_path)

# get generated csv
files = dbutils.fs.ls(temp_path)
csv_file = [f.path for f in files if f.path.endswith(".csv")][0]

# move to final location
final_path = f"{base_path}/{file_name}.csv"
dbutils.fs.mv(csv_file, final_path)

# cleanup temp folder
dbutils.fs.rm(temp_path, True)

print(f"Created file: {final_path}")

In [0]:
# # get generated file
# files = dbutils.fs.ls(path)
# csv_file = [f.path for f in files if f.path.endswith(".csv")][0]

# # rename to dataset1.csv
# dbutils.fs.mv(csv_file, path + "/dataset1.csv")